# 01 · Auditar un export de Label Studio

Este notebook toma un export CSV de Label Studio y crea artefactos locales para revisar su calidad. No entrena modelos, no modifica anotaciones y no crea el conjunto final automáticamente.

Usa CSV: el export de Label Studio ya está aplanado y esta etapa necesita una tabla revisable. No se requiere JSONL.

## 1. Ubicar el export local

Copia el archivo descargado desde Label Studio a `artifacts/labelstudio/exports/`, sin cambiarle el nombre. El notebook detecta automáticamente el CSV más reciente de esa carpeta. Está ignorada por Git: no publiques cuerpos de noticias ni anotaciones.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

# Así cada persona puede dejar su CSV exportado sin escribir un nombre fechado en el código.
EXPORT_DIR = ROOT / 'artifacts' / 'labelstudio' / 'exports'
export_files = sorted(EXPORT_DIR.glob('*.csv'), key=lambda path: path.stat().st_mtime, reverse=True)
if not export_files:
    raise FileNotFoundError(f'No hay export CSV en: {EXPORT_DIR}')
EXPORT_PATH = export_files[0]

print(f'Export seleccionado: {EXPORT_PATH.name}')
if len(export_files) > 1:
    print(f'Aviso: hay {len(export_files)} CSV; se usará el más reciente.')

## 2. Leer y validar el esquema

El export debe contener datos de la noticia, elegibilidad y `reference_claim`. Si falta una columna, el notebook se detiene antes de producir archivos derivados.

In [ ]:
import csv
import json
from collections import Counter

REQUIRED_COLUMNS = {
    'annotation_id', 'record_id', 'title', 'body', 'news_eligibility', 'reference_claim',
}
with EXPORT_PATH.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    export_columns = reader.fieldnames or []
    raw_rows = list(reader)

missing_columns = sorted(REQUIRED_COLUMNS.difference(export_columns))
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {', '.join(missing_columns)}")
if not raw_rows:
    raise ValueError('El export no contiene tareas.')

print(f'Filas exportadas: {len(raw_rows)}')
print('Columnas:', ', '.join(export_columns))

## 3. Auditar elegibilidad, claims y duplicados

Una fila candidata debe tener anotación, `APTA_CON_CLAIM`, título, cuerpo y `reference_claim`. Los duplicados se señalan para revisión; no se eliminan automáticamente. El resumen también indica si ya hay suficientes candidatas para congelar el conjunto de 500 claims del notebook 02.

In [ ]:
# Debe coincidir con TARGET_ROWS del notebook 02.
TARGET_REFERENCE_CLAIMS = 500

def clean_text(value):
    return ' '.join(str(value or '').split())

def normalize_choice(value):
    text = clean_text(value)
    if text.startswith('[') and text.endswith(']'):
        try:
            decoded = json.loads(text)
            if isinstance(decoded, list) and decoded:
                return clean_text(decoded[0])
        except json.JSONDecodeError:
            pass
    return text

# Solo una noticia elegible con un claim de referencia pasa al conjunto candidato.
record_counts = Counter(clean_text(row.get('record_id')) for row in raw_rows if clean_text(row.get('record_id')))
audited_rows = []
for row in raw_rows:
    audit = dict(row)
    record_id = clean_text(row.get('record_id'))
    eligibility = normalize_choice(row.get('news_eligibility'))
    reference_claim = clean_text(row.get('reference_claim'))
    flags = []
    if not clean_text(row.get('annotation_id')):
        flags.append('SIN_ANOTACION')
    if eligibility != 'APTA_CON_CLAIM':
        flags.append('NO_APTA_CON_CLAIM')
    if not clean_text(row.get('title')):
        flags.append('SIN_TITULO')
    if not clean_text(row.get('body')):
        flags.append('SIN_CUERPO')
    if not reference_claim:
        flags.append('SIN_REFERENCE_CLAIM')
    if record_id and record_counts[record_id] > 1:
        flags.append('RECORD_ID_DUPLICADO')
    audit.update({
        'news_eligibility_normalized': eligibility,
        'reference_claim_normalized': reference_claim,
        'quality_flags': '|'.join(flags),
        'is_reference_candidate': 'SI' if not flags else 'NO',
    })
    audited_rows.append(audit)

eligibility_counts = Counter(row['news_eligibility_normalized'] or '<vacío>' for row in audited_rows)
candidate_rows = [row for row in audited_rows if row['is_reference_candidate'] == 'SI']
print('Elegibilidad:', dict(sorted(eligibility_counts.items())))
print(f'Candidatas sin banderas: {len(candidate_rows)}')
print(f'record_id duplicados: {sum(1 for count in record_counts.values() if count > 1)} grupos')
if len(candidate_rows) < TARGET_REFERENCE_CLAIMS:
    missing = TARGET_REFERENCE_CLAIMS - len(candidate_rows)
    print(f'Faltan {missing} candidatas aptas para construir el conjunto de {TARGET_REFERENCE_CLAIMS}.')
else:
    print(f'Hay suficientes candidatas para construir el conjunto de {TARGET_REFERENCE_CLAIMS}.')

## 4. Escribir los CSV locales de auditoría

`labelstudio_export_audit.csv` contiene todas las filas y banderas. `reference_claims_candidates.csv` contiene las candidatas limpias, pero aún no es el conjunto final: antes hay que revisar duplicados y decidir el tamaño/muestreo final. Los resultados se actualizan en la carpeta estable `processed/current_export/`.

In [ ]:
import hashlib

OUTPUT_DIR = ROOT / 'artifacts' / 'labelstudio' / 'processed' / 'current_export'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audit_path = OUTPUT_DIR / 'labelstudio_export_audit.csv'
audit_columns = export_columns + ['news_eligibility_normalized', 'reference_claim_normalized', 'quality_flags', 'is_reference_candidate']
with audit_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=audit_columns, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(audited_rows)

candidate_columns = [
    'record_id', 'title', 'body', 'reference_claim_normalized', 'country', 'requested_country',
    'source_name', 'canonical_url', 'published_at', 'annotation_id',
]
candidates_path = OUTPUT_DIR / 'reference_claims_candidates.csv'
with candidates_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=candidate_columns, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(candidate_rows)

summary = {
    'source_export_filename': EXPORT_PATH.name,
    'source_export_sha256': hashlib.sha256(EXPORT_PATH.read_bytes()).hexdigest(),
    'export_rows': len(audited_rows),
    'reference_candidates': len(candidate_rows),
    'unique_record_ids': len(record_counts),
    'duplicate_record_id_groups': sum(1 for count in record_counts.values() if count > 1),
    'eligibility_counts': dict(sorted(eligibility_counts.items())),
}
summary_path = OUTPUT_DIR / 'audit_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Auditoría: {audit_path.relative_to(ROOT)}')
print(f'Candidatas: {candidates_path.relative_to(ROOT)}')
print(f'Resumen: {summary_path.relative_to(ROOT)}')

## Siguiente decisión

Revisa primero las filas con `quality_flags` y los `record_id` duplicados. Solo después se construye un manifiesto final de referencias y sus particiones de entrenamiento, validación y prueba.